## 1. Thiết lập Môi trường và Tải Cấu hình (Soft-Attention)
Trong phần này, chúng ta sẽ tải trọng số mô hình tốt nhất (`BEST_checkpoint.pth.tar`) đã được huấn luyện với kiến trúc ResNet-101 kết hợp LSTM và cơ chế **Bahdanau Soft-Attention**.

In [1]:
import torch
import torch.backends.cudnn as cudnn
import torch.nn.functional as F
import torchvision.transforms as transforms
from nltk.translate.bleu_score import corpus_bleu
import json
import yaml
import h5py
import os
from tqdm.notebook import tqdm

# Import kiến trúc Mạng Đề Xuất (Có Attention)
from src.models.Resnet101 import Encoder
from src.models.Decoder import DecoderWithAttention

# 1. Khởi tạo thiết bị - Ưu tiên sử dụng RTX 3060
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cudnn.benchmark = True
print(f"✅ Đang chạy thực nghiệm trên thiết bị: {device}")
if torch.cuda.is_available():
    print(f"🚀 GPU detected: {torch.cuda.get_device_name(0)}")

# 2. Tải cấu hình
with open('configs/default.yaml', 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

# 3. Trích xuất đường dẫn
data_folder = cfg['dataset']['data_folder']
data_name = cfg['dataset']['data_name']
checkpoint_path = cfg['evaluation']['checkpoint_path']
beam_size = cfg['evaluation']['top_k']

word_map_file = os.path.join(data_folder, f'WORDMAP_{data_name}.json')
test_image_path = os.path.join(data_folder, f'TEST_IMAGES_{data_name}.hdf5')
test_caps_path = os.path.join(data_folder, f'TEST_CAPTIONS_{data_name}.json')

# 4. Tham số mạng Nơ-ron
emb_dim, attention_dim, decoder_dim, dropout = 512, 512, 512, 0.5

✅ Đang chạy thực nghiệm trên thiết bị: cuda
🚀 GPU detected: NVIDIA GeForce RTX 3060


## 2. Hàm Suy luận bằng Beam Search (Soft-Attention)
Đoạn mã này thực thi quá trình giải mã (Decoding) sử dụng thuật toán Beam Search. Điểm khác biệt cốt lõi nằm ở dòng `decoder.attention(encoder_out, h_state)` — nơi mô hình tính toán trọng số $\alpha$ để quyết định tập trung vào vùng ảnh nào trước khi sinh từ tiếp theo.

In [2]:
def evaluate_attention_model(beam_size):
    # 1. Tải Word Map
    with open(word_map_file, 'r') as j:
        word_map = json.load(j)
    vocab_size = len(word_map)

    # 2. Tải trọng số mô hình
    print(f"[*] Đang tải checkpoint từ: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

    encoder = Encoder().to(device)
    encoder.load_state_dict(checkpoint['encoder'])
    encoder.eval()

    decoder = DecoderWithAttention(attention_dim=attention_dim, embed_dim=emb_dim,
                                   decoder_dim=decoder_dim, vocab_size=vocab_size,
                                   dropout=dropout).to(device)

    # Xử lý lệch tên biến Attention
    state_dict = checkpoint['decoder']
    new_state_dict = {k.replace('att.', 'attention.') if k.startswith('att.') else k: v for k, v in state_dict.items()}
    decoder.load_state_dict(new_state_dict, strict=False)
    decoder.eval()
    print("✅ Đã xử lý xong việc lệch tên biến Attention và load thành công!")

    # 3. Tải Dữ liệu Test sử dụng context manager để tránh file lock trên Windows
    with h5py.File(test_image_path, 'r') as h:
        images = h['images']
        with open(test_caps_path, 'r') as j:
            captions = json.load(j)

        references, hypotheses = [], []
        normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

        print(f"[*] Khởi chạy Beam Search (k={beam_size}) trên {len(images)} ảnh...")
        
        for i in tqdm(range(len(images)), desc="Evaluating Attention Model"):
            k = beam_size
            # Đẩy ảnh lên GPU trực tiếp
            image = torch.FloatTensor(images[i]).unsqueeze(0).to(device) / 255.
            image = normalize(image)

            with torch.no_grad():
                encoder_out = encoder(image)
            
            encoder_dim = encoder_out.size(-1)
            encoder_out = encoder_out.view(1, -1, encoder_dim).expand(k, -1, encoder_dim)

            k_prev_words = torch.full((k, 1), word_map['<start>'], dtype=torch.long, device=device)
            seqs = k_prev_words
            top_k_scores = torch.zeros(k, 1, device=device)
            h_state, c_state = decoder.init_hidden_state(encoder_out)

            step, complete_seqs, complete_seqs_scores = 1, [], []

            while True:
                embeddings = decoder.embedding(k_prev_words).squeeze(1)
                awe, _ = decoder.attention(encoder_out, h_state)
                gate = decoder.sigmoid(decoder.f_beta(h_state))
                awe = gate * awe

                h_state, c_state = decoder.decode_step(torch.cat([embeddings, awe], dim=1), (h_state, c_state))
                scores = F.log_softmax(decoder.fc(h_state), dim=1)
                scores = top_k_scores.expand_as(scores) + scores

                if step == 1:
                    top_k_scores, top_k_words = scores[0].topk(k, 0, True, True)
                else:
                    top_k_scores, top_k_words = scores.view(-1).topk(k, 0, True, True)

                prev_word_inds = top_k_words // vocab_size
                next_word_inds = top_k_words % vocab_size

                seqs = torch.cat([seqs[prev_word_inds], next_word_inds.unsqueeze(1)], dim=1)
                incomplete_inds = [ind for ind, next_word in enumerate(next_word_inds) if next_word != word_map['<end>']]
                complete_inds = list(set(range(len(next_word_inds))) - set(incomplete_inds))

                if len(complete_inds) > 0:
                    complete_seqs.extend(seqs[complete_inds].tolist())
                    complete_seqs_scores.extend(top_k_scores[complete_inds].cpu().tolist())

                k -= len(complete_inds)
                if k == 0 or step > 50: break

                # Cập nhật trạng thái cho bước tiếp theo (Fix lỗi NoneType)
                seqs = seqs[incomplete_inds]
                h_state = h_state[prev_word_inds[incomplete_inds]]
                c_state = c_state[prev_word_inds[incomplete_inds]]
                encoder_out = encoder_out[prev_word_inds[incomplete_inds]] 
                top_k_scores = top_k_scores[incomplete_inds].unsqueeze(1)
                k_prev_words = next_word_inds[incomplete_inds].unsqueeze(1)
                step += 1

            # Lấy chuỗi tốt nhất
            seq = complete_seqs[complete_seqs_scores.index(max(complete_seqs_scores))] if complete_seqs_scores else seqs[0].tolist()
            
            # Xử lý kết quả
            img_caps = captions[i * 5:(i * 5) + 5]
            references.append([[w for w in c if w not in {word_map['<start>'], word_map['<end>'], word_map['<pad>']}] for c in img_caps])
            hypotheses.append([w for w in seq if w not in {word_map['<start>'], word_map['<end>'], word_map['<pad>']}])

    return references, hypotheses

## 3. Tổng hợp Kết quả Đánh giá
Đoạn mã này thực thi hàm suy luận và tính toán các chỉ số **BLEU-1 đến BLEU-4** của mô hình Đề xuất. Các chỉ số này sẽ được tổng hợp để chứng minh hiệu quả vượt trội của cơ chế Attention so với Baseline.

In [3]:
# Gọi hàm suy luận
refs, hypos = evaluate_attention_model(beam_size)

print("\n[*] Đang tính toán các chỉ số n-gram BLEU...")

# Tính toán BLEU scores
bleu1 = corpus_bleu(refs, hypos, weights=(1.0, 0, 0, 0))
bleu2 = corpus_bleu(refs, hypos, weights=(0.5, 0.5, 0, 0))
bleu3 = corpus_bleu(refs, hypos, weights=(0.33, 0.33, 0.33, 0))
bleu4 = corpus_bleu(refs, hypos, weights=(0.25, 0.25, 0.25, 0.25))

print("\n" + "═"*54)
print(f"🎉 BÁO CÁO KẾT QUẢ MÔ HÌNH ĐỀ XUẤT (SOFT-ATTENTION)")
print("═"*54)
print(f"➤ Thiết bị tính toán     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"➤ Kích thước Beam Search : {beam_size}")
print(f"➤ Tập dữ liệu Test       : {len(hypos)} ảnh")
print("-" * 54)
print(f"➤ BLEU-1 Score           : {bleu1 * 100:>6.2f}%")
print(f"➤ BLEU-2 Score           : {bleu2 * 100:>6.2f}%")
print(f"➤ BLEU-3 Score           : {bleu3 * 100:>6.2f}%")
print(f"➤ BLEU-4 Score           : {bleu4 * 100:>6.2f}%")
print("═"*54 + "\n")

[*] Đang tải checkpoint từ: models/BEST_checkpoint_coco_5_cap_per_img_5_min_word_freq.pth.tar


C:\Users\leotran\OneDrive\Desktop\ThS_PTIT_HTTT\AI\Final_term\Final\env_gpu\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\leotran\OneDrive\Desktop\ThS_PTIT_HTTT\AI\Final_term\Final\env_gpu\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


✅ Đã xử lý xong việc lệch tên biến Attention và load thành công!
[*] Khởi chạy Beam Search (k=5) trên 5000 ảnh...


Evaluating Attention Model:   0%|          | 0/5000 [00:00<?, ?it/s]


[*] Đang tính toán các chỉ số n-gram BLEU...

══════════════════════════════════════════════════════
🎉 BÁO CÁO KẾT QUẢ MÔ HÌNH ĐỀ XUẤT (SOFT-ATTENTION)
══════════════════════════════════════════════════════
➤ Thiết bị tính toán     : NVIDIA GeForce RTX 3060
➤ Kích thước Beam Search : 5
➤ Tập dữ liệu Test       : 5000 ảnh
------------------------------------------------------
➤ BLEU-1 Score           :  70.93%
➤ BLEU-2 Score           :  53.69%
➤ BLEU-3 Score           :  40.54%
➤ BLEU-4 Score           :  30.27%
══════════════════════════════════════════════════════

